# Xception Model Training for NIH Chest X-ray Dataset

This notebook trains an Xception model for chest X-ray classification on the prepared NIH dataset.

## Prerequisites

**Important**: Make sure you have completed `playbook1_prepare_dataset.ipynb` first to create the required datasets.

## Overview

This notebook will:
- Train an Xception model with ImageNet pretrained weights
- Use the small dataset for faster development (or full dataset for production)
- Save the best model weights and training metrics
- Provide comprehensive evaluation and visualization

## Training Configuration

- **Model**: Xception (ImageNet pretrained)
- **Epochs**: 50 (with early stopping)
- **Batch Size**: 32
- **Learning Rate**: 1e-4
- **Early Stopping**: 10 epochs patience
- **Optimizer**: AdamW with weight decay
- **Scheduler**: CosineAnnealingWarmRestarts

## Expected Output

After training, you'll have:
- Trained model weights (`models/xception/xception_best.pth`)
- Training performance metrics (CSV file)
- Test evaluation results (JSON file)
- Confusion matrix visualization
- Training curves and plots

---

**Ready to train!** Run the cells below to start training.


## Setup and Imports


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

print(f"Project root: {project_root}")
print(f"Current working directory: {os.getcwd()}")
print("Xception training setup complete!")


## Dataset Selection

Check available datasets and select the appropriate one for training.


In [ ]:
print("Dataset Selection")
print("=" * 50)

# Check available datasets
small_dataset_path = project_root / "dataset" / "NIH-Chest-X-ray" / "Classification_SMALL"
full_dataset_path = project_root / "dataset" / "NIH-Chest-X-ray" / "Classification"

print("Checking available datasets...")

if small_dataset_path.exists():
    print(f"✓ Small dataset found: {small_dataset_path}")
    # Count images in small dataset
    small_count = len(list(small_dataset_path.rglob("*.jpg")))
    print(f"  Images: {small_count:,}")
else:
    print(f"✗ Small dataset not found: {small_dataset_path}")

if full_dataset_path.exists():
    print(f"✓ Full dataset found: {full_dataset_path}")
    # Count images in full dataset
    full_count = len(list(full_dataset_path.rglob("*.jpg")))
    print(f"  Images: {full_count:,}")
else:
    print(f"✗ Full dataset not found: {full_dataset_path}")

# Select dataset for training
if small_dataset_path.exists():
    dataset_path = small_dataset_path
    print(f"\nSelected dataset: SMALL (recommended for development)")
    print("Benefits: Faster training, good for testing and iteration")
elif full_dataset_path.exists():
    dataset_path = full_dataset_path
    print(f"\nSelected dataset: FULL (recommended for production)")
    print("Benefits: Better performance, longer training time")
else:
    print("\n❌ No suitable dataset found!")
    print("Please run playbook1_prepare_dataset.ipynb first to create the datasets.")
    dataset_path = None

if dataset_path:
    print(f"\nTraining will use: {dataset_path}")
    print("Proceed to the next cell to start training.")
else:
    print("\nCannot proceed without a dataset. Please check the dataset preparation.")


## Start Xception Training

Train the Xception model on the selected dataset.


In [ ]:
if dataset_path:
    print("Starting Xception Training")
    print("=" * 50)
    
    print("Training Configuration:")
    print("  - Model: Xception (ImageNet pretrained)")
    print("  - Epochs: 50 (with early stopping)")
    print("  - Batch Size: 32")
    print("  - Learning Rate: 1e-4")
    print("  - Early Stopping Patience: 10")
    print("  - Optimizer: AdamW with weight decay")
    print("  - Scheduler: CosineAnnealingWarmRestarts")
    
    print(f"\nDataset: {dataset_path}")
    print("\nTraining may take 1-3 hours depending on your hardware...")
    print("Progress will be shown below:")
    print("-" * 50)
    
    # Record start time
    start_time = time.time()
    
    # Run Xception training script
    result = subprocess.run([
        "uv", "run", "python", 
        str(project_root / "src" / "train" / "train_Xception.py"),
        str(dataset_path),
        "--max-epochs", "50",
        "--batch-size", "32",
        "--learning-rate", "1e-4",
        "--early-stopping-patience", "10"
    ], capture_output=True, text=True)
    
    # Calculate training time
    training_time = time.time() - start_time
    
    print("\n" + "=" * 50)
    print("TRAINING OUTPUT:")
    print("=" * 50)
    print(result.stdout)
    
    if result.stderr:
        print("\nSTDERR:")
        print("=" * 50)
        print(result.stderr)
    
    print("\n" + "=" * 50)
    print("TRAINING SUMMARY:")
    print("=" * 50)
    print(f"Training completed in {training_time/3600:.2f} hours")
    
    if result.returncode == 0:
        print("✓ Xception training completed successfully!")
        print("Model and results saved to: models/xception/")
    else:
        print(f"✗ Xception training failed with return code: {result.returncode}")
        print("Please check the error messages above.")
        
else:
    print("Cannot start training - no dataset available.")
    print("Please run playbook1_prepare_dataset.ipynb first.")


## Training Results Analysis

Analyze the training results and display performance metrics.


In [ ]:
print("Training Results Analysis")
print("=" * 50)

models_dir = project_root / "models" / "xception"

if models_dir.exists():
    print(f"Model directory: {models_dir}")
    print("\nGenerated files:")
    
    for file_path in models_dir.iterdir():
        if file_path.is_file():
            file_size = file_path.stat().st_size
            if file_size > 1024*1024:  # > 1MB
                size_str = f"{file_size/(1024*1024):.1f} MB"
            else:
                size_str = f"{file_size/1024:.1f} KB"
            print(f"  - {file_path.name}: {size_str}")
    
    # Check for training performance CSV
    csv_files = list(models_dir.glob("*_training_performance.csv"))
    if csv_files:
        print(f"\nFound {len(csv_files)} training performance file(s)")
        
        # Load and display the latest training performance
        latest_csv = max(csv_files, key=lambda x: x.stat().st_mtime)
        print(f"Latest training performance file: {latest_csv.name}")
        
        try:
            df = pd.read_csv(latest_csv)
            print(f"\nTraining Summary:")
            print(f"  Total epochs: {len(df)}")
            print(f"  Best validation F1: {df['val_f1'].max():.4f}")
            print(f"  Best validation accuracy: {df['val_accuracy'].max():.2f}%")
            print(f"  Final training loss: {df['train_loss'].iloc[-1]:.4f}")
            print(f"  Final validation loss: {df['val_loss'].iloc[-1]:.4f}")
            
            # Show last few epochs
            print(f"\nLast 5 epochs:")
            print(df[['epoch', 'train_loss', 'val_loss', 'train_accuracy', 'val_accuracy', 'val_f1']].tail())
            
        except Exception as e:
            print(f"Error reading training performance file: {e}")
    
    # Check for test results
    test_results_file = models_dir / "test_results.json"
    if test_results_file.exists():
        print(f"\nTest Results Available: {test_results_file.name}")
        
        try:
            with open(test_results_file, 'r') as f:
                test_results = json.load(f)
            
            print(f"\nTest Results:")
            print(f"  Test Accuracy: {test_results['accuracy']:.2f}%")
            if test_results['macro_auc']:
                print(f"  Macro AUC: {test_results['macro_auc']:.4f}")
            
            # Show per-class results
            if 'classification_report' in test_results:
                report = test_results['classification_report']
                print(f"\nPer-class F1 scores:")
                for class_name, metrics in report.items():
                    if isinstance(metrics, dict) and 'f1-score' in metrics:
                        print(f"  {class_name}: {metrics['f1-score']:.4f}")
                        
        except Exception as e:
            print(f"Error reading test results: {e}")
    
    # Check for confusion matrix
    confusion_matrix_file = models_dir / "confusion_matrix.png"
    if confusion_matrix_file.exists():
        print(f"\nConfusion Matrix: {confusion_matrix_file.name}")
        
        # Display the confusion matrix
        try:
            from IPython.display import Image, display
            display(Image(str(confusion_matrix_file)))
        except ImportError:
            print("Confusion matrix image available but cannot display in this environment.")
            
else:
    print("Model directory not found. Training may not have completed successfully.")


## Training Performance Visualization

Create visualizations of the training performance.


In [ ]:
# Plot training curves if performance data is available
models_dir = project_root / "models" / "xception"
csv_files = list(models_dir.glob("*_training_performance.csv")) if models_dir.exists() else []

if csv_files:
    latest_csv = max(csv_files, key=lambda x: x.stat().st_mtime)
    df = pd.read_csv(latest_csv)
    
    print("Creating Training Performance Visualizations")
    print("=" * 50)
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Xception Training Performance', fontsize=16)
    
    # Plot 1: Loss curves
    axes[0, 0].plot(df['epoch'], df['train_loss'], label='Training Loss', color='blue')
    axes[0, 0].plot(df['epoch'], df['val_loss'], label='Validation Loss', color='red')
    axes[0, 0].set_title('Loss Curves')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Plot 2: Accuracy curves
    axes[0, 1].plot(df['epoch'], df['train_accuracy'], label='Training Accuracy', color='blue')
    axes[0, 1].plot(df['epoch'], df['val_accuracy'], label='Validation Accuracy', color='red')
    axes[0, 1].set_title('Accuracy Curves')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Plot 3: F1 Score curves
    axes[1, 0].plot(df['epoch'], df['train_f1'], label='Training F1', color='blue')
    axes[1, 0].plot(df['epoch'], df['val_f1'], label='Validation F1', color='red')
    axes[1, 0].set_title('F1 Score Curves')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1 Score (%)')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Plot 4: Learning rate
    axes[1, 1].plot(df['epoch'], df['learning_rate'], color='green')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print("Training performance visualization completed!")
    
    # Save the plot
    plot_path = models_dir / "training_curves.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"Training curves saved to: {plot_path}")
    
else:
    print("No training performance data found. Training may not have completed successfully.")


## Next Steps

Congratulations! You have successfully trained an Xception model for chest X-ray classification.

### What You've Accomplished

1. **Trained Xception Model** - Fine-tuned on NIH Chest X-ray dataset
2. **Generated Comprehensive Results** - Training metrics, test evaluation, and visualizations
3. **Saved Model Assets** - Best weights, class mappings, and configuration files

### Generated Files

Your trained Xception model and results are saved in:
- `models/xception/xception_best.pth` - Best model weights
- `models/xception/xception_latest.pth` - Latest checkpoint
- `models/xception/{timestamp}_training_performance.csv` - Training metrics per epoch
- `models/xception/test_results.json` - Test evaluation results
- `models/xception/confusion_matrix.png` - Confusion matrix visualization
- `models/xception/training_curves.png` - Training performance plots
- `models/xception/class_indices.json` - Class name to index mapping
- `models/xception/training_history.json` - Complete training history
- `models/xception/tensorboard_logs/` - TensorBoard log directory

### Further Development Options

#### 1. Generate Heatmaps
Use the trained model with Grad-CAM++ to generate heatmaps for model interpretability:

```python
# Example usage (implement generate_heatmap.py)
from generate_heatmap import generate_gradcam_heatmap

heatmap = generate_gradcam_heatmap(
    model_path="models/xception/xception_best.pth",
    image_path="path/to/chest_xray.jpg",
    class_indices_path="models/xception/class_indices.json"
)
```

#### 2. Train Other Models
Compare performance with other architectures:

```bash
# ResNet50
uv run python src/train/train_resnet50.py dataset/NIH-Chest-X-ray/Classification_SMALL --max-epochs 50

# DenseNet121
uv run python src/train/train_DenseNet121.py dataset/NIH-Chest-X-ray/Classification_SMALL --max-epochs 50

# DenseNet169
uv run python src/train/train_DenseNet169.py dataset/NIH-Chest-X-ray/Classification_SMALL --max-epochs 50

# InceptionResNetV2
uv run python src/train/train_InceptionResNetV2.py dataset/NIH-Chest-X-ray/Classification_SMALL --max-epochs 50
```

#### 3. Full Dataset Training
For production use, train on the complete dataset:

```bash
uv run python src/train/train_Xception.py dataset/NIH-Chest-X-ray/Classification --max-epochs 100 --batch-size 16
```

#### 4. Hyperparameter Tuning
Experiment with different configurations:

```bash
# Different learning rates
uv run python src/train/train_Xception.py dataset/NIH-Chest-X-ray/Classification_SMALL --learning-rate 5e-5 --max-epochs 50

# Different batch sizes
uv run python src/train/train_Xception.py dataset/NIH-Chest-X-ray/Classification_SMALL --batch-size 16 --max-epochs 50

# Different dropout rates
uv run python src/train/train_Xception.py dataset/NIH-Chest-X-ray/Classification_SMALL --dropout-rate 0.3 --max-epochs 50
```

#### 5. Model Ensemble
Combine multiple models for improved accuracy:

```python
# Example ensemble approach
models = [
    "models/xception/xception_best.pth",
    "models/resnet50/resnet50_best.pth",
    "models/densenet121/densenet121_best.pth"
]
# Implement ensemble prediction logic
```

### TensorBoard Analysis

View detailed training logs in TensorBoard:

```bash
uv run tensorboard --logdir models/xception/tensorboard_logs
```

Then open http://localhost:6006 in your browser to explore:
- Loss and accuracy curves
- Learning rate schedule
- Per-class metrics
- Model architecture visualization
- Sample predictions with images

### Model Deployment

Your trained model is ready for:
- **Inference**: Load `xception_best.pth` for predictions
- **Grad-CAM++**: Generate heatmaps for interpretability
- **Production**: Deploy to cloud platforms or edge devices
- **Research**: Use for further analysis and experimentation

---

**Training completed successfully!** Your Xception model is ready for chest X-ray classification and heatmap generation.
